# Kaggle S6E3 — Customer Churn v2
Added: ORIG dataset stats + DIGIT features + ROUND features (no slow TE interactions)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
SEED = 42; N_SPLITS = 5
np.random.seed(SEED)
print('ok')

In [ ]:
import os
if os.path.exists('/kaggle/input'):
    TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
    TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
    SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv'
    ORIG_PATH  = '/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'
else:
    TRAIN_PATH = 'data/train.csv'
    TEST_PATH  = 'data/test.csv'
    SUB_PATH   = 'data/sample_submission.csv'
    ORIG_PATH  = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH)
sub   = pd.read_csv(SUB_PATH)
print('train', train.shape, '| test', test.shape, '| orig', orig.shape)

In [ ]:
TARGET='Churn'
CATS=['gender','SeniorCitizen','Partner','Dependents','PhoneService',
    'MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']

train_ids=train['id'].copy(); test_ids=test['id'].copy()
train=train.drop(columns=['id']); test=test.drop(columns=['id'])
if 'customerID' in orig.columns: orig=orig.drop(columns=['customerID'])

def map_target(s): return s.map({'Yes':1,'No':0}).astype('int8') if s.dtype==object else s.astype('int8')
train[TARGET]=map_target(train[TARGET]); orig[TARGET]=map_target(orig[TARGET])

for df in [train,test,orig]:
    df['tenure']=pd.to_numeric(df['tenure'],errors='coerce').astype('float32')
    df['MonthlyCharges']=pd.to_numeric(df['MonthlyCharges'],errors='coerce').astype('float32')
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'].astype(str).str.strip().replace('',np.nan),errors='coerce').astype('float32')
    m=df['TotalCharges'].isna()
    df.loc[m,'TotalCharges']=df.loc[m,'tenure']*df.loc[m,'MonthlyCharges']
    for c in CATS:
        if c in df.columns: df[c]=df[c].astype(str).fillna('Missing')

BASE=[c for c in train.columns if c!=TARGET]
print('BASE cols:',len(BASE),'| churn rate:',round(train[TARGET].mean(),4))

In [ ]:
from itertools import combinations

# ── DIGIT features ────────────────────────────────────────────
def add_digit_cols(df, col, multiplier):
    if multiplier == 'direct':
        vals = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(int)
    else:
        vals = (pd.to_numeric(df[col], errors='coerce') * multiplier).round(0).fillna(-1).astype(int)
    max_len = {'tenure':3,'MonthlyCharges':5,'TotalCharges':6}.get(col, vals.astype(str).str.len().max())
    s = vals.astype(str).str.zfill(max_len)
    created = []
    for i in range(max_len):
        n = f'{col}_DIGIT_{i+1}'; df[n] = s.str[i].astype(int); created.append(n)
    return created

DIGIT = []
for df in [train, test]:
    d = add_digit_cols(df, 'tenure', 'direct')
    add_digit_cols(df, 'MonthlyCharges', 100)
    add_digit_cols(df, 'TotalCharges', 1)
    if df is train:
        DIGIT = d + [f'MonthlyCharges_DIGIT_{i+1}' for i in range(5)] + [f'TotalCharges_DIGIT_{i+1}' for i in range(6)]
print(len(DIGIT), 'DIGIT features')

# ── ROUND features ────────────────────────────────────────────
ROUND = []
for col, levels in [('MonthlyCharges',[('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('TotalCharges',  [('1s',0),('10s',-1),('100s',-2),('1000s',-3)]),
                    ('tenure',        [('1s',0),('10s',-1)])]:
    for suffix, level in levels:
        n = f'{col}_ROUND_{suffix}'; ROUND.append(n)
        for df in [train, test]: df[n] = df[col].round(level).fillna(-1).astype(int)
print(len(ROUND), 'ROUND features')

# ── ORIG dataset stats ────────────────────────────────────────
ORIG_FEATS = []
orig_mean = float(orig[TARGET].mean())
for col in BASE:
    mm = orig.groupby(col)[TARGET].mean()
    cm = orig.groupby(col).size()
    for df in [train, test]:
        df[f'orig_mean_{col}']  = df[col].map(mm).fillna(orig_mean)
        df[f'orig_count_{col}'] = df[col].map(cm).fillna(0)
    ORIG_FEATS += [f'orig_mean_{col}', f'orig_count_{col}']
print(len(ORIG_FEATS), 'ORIG features')

# ── INTERACTION features (string concat, TE inside CV) ────────
INTER = []
for col1, col2 in combinations(BASE, 2):
    n = f'{col1}_{col2}'; INTER.append(n)
    for df in [train, test]: df[n] = df[col1].astype(str) + '_' + df[col2].astype(str)
print(len(INTER), 'INTER features')

FEATURES = list(dict.fromkeys(BASE + ORIG_FEATS + INTER + ROUND + DIGIT))
print(len(FEATURES), 'total features')

In [ ]:
from sklearn.model_selection import KFold
import time

def factorize3(tr, val, te):
    codes, _ = pd.factorize(pd.concat([tr, val, te], ignore_index=True).astype(str))
    n1, n2 = len(tr), len(val)
    return (pd.Series(codes[:n1], index=tr.index, dtype='int32'),
            pd.Series(codes[n1:n1+n2], index=val.index, dtype='int32'),
            pd.Series(codes[n1+n2:], index=te.index, dtype='int32'))

# ── Precompute TE on interactions ONCE (outside CV) ───────────
print('Precomputing target encoding for 171 interactions...')
t0 = time.time()
X_all = train[FEATURES].copy()
y_all = train[TARGET].copy()
Xtest_raw = test[FEATURES].copy()
gm = float(y_all.mean())

oof_te = pd.DataFrame(index=X_all.index)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_i, (ti, vi) in enumerate(kf.split(X_all), 1):
    Xt = X_all.iloc[ti]; yt = y_all.iloc[ti]; Xv = X_all.iloc[vi]
    fm = float(yt.mean())
    for col in INTER:
        mp = Xt.assign(_y=yt.values).groupby(col)['_y'].mean()
        oof_te.loc[Xv.index, f'TE_{col}'] = Xv[col].map(mp).fillna(fm).values
    print(f'  TE fold {fold_i}/5 done — {time.time()-t0:.0f}s elapsed')

print('Computing test TE...')
full_map = {col: X_all.assign(_y=y_all.values).groupby(col)['_y'].mean() for col in INTER}
test_te = pd.DataFrame({f'TE_{col}': Xtest_raw[col].map(full_map[col]).fillna(gm) for col in INTER},
                       index=Xtest_raw.index)

X_all = X_all.drop(columns=INTER)
Xtest_raw = Xtest_raw.drop(columns=INTER)
for col in INTER:
    X_all[f'TE_{col}']     = oof_te[f'TE_{col}']
    Xtest_raw[f'TE_{col}'] = test_te[f'TE_{col}']

print(f'TE done in {time.time()-t0:.0f}s. Final feature count: {X_all.shape[1]}')

# ── LightGBM — multiple seeds (Technique 7) ──────────────────
LGB_SEEDS = [42, 123, 456]
lgb_params = {
    'objective':'binary','metric':'auc','n_estimators':2000,'learning_rate':0.05,
    'num_leaves':127,'subsample':0.8,'colsample_bytree':0.6,'min_child_samples':20,
    'reg_alpha':0.1,'reg_lambda':1.0,'n_jobs':-1,'verbose':-1
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof   = np.zeros(len(X_all),    dtype=np.float64)
tpred = np.zeros(len(Xtest_raw),dtype=np.float64)

for seed in LGB_SEEDS:
    print(f'\n=== LightGBM seed={seed} ===')
    oof_s = np.zeros(len(X_all), dtype=np.float64)
    tp_s  = np.zeros(len(Xtest_raw), dtype=np.float64)
    for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
        fold_start = time.time()
        Xtr = X_all.iloc[tri].copy(); Xval = X_all.iloc[vali].copy()
        ytr = y_all.iloc[tri].copy(); yval = y_all.iloc[vali].copy()
        Xte = Xtest_raw.copy()
        for c in CATS:
            if c in Xtr.columns: Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        for c in Xtr.select_dtypes('object').columns:
            Xtr[c], Xval[c], Xte[c] = factorize3(Xtr[c], Xval[c], Xte[c])
        params = {**lgb_params, 'random_state': seed}
        model = lgb.LGBMClassifier(**params)
        model.fit(Xtr, ytr, eval_set=[(Xval, yval)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])
        oof_s[vali] = model.predict_proba(Xval)[:, 1]
        tp_s += model.predict_proba(Xte)[:, 1] / N_SPLITS
        print(f'  Fold {fold} AUC: {roc_auc_score(yval, oof_s[vali]):.5f}  '
              f'iter={model.best_iteration_}  time={time.time()-fold_start:.0f}s')
    print(f'  Seed {seed} OOF AUC: {roc_auc_score(y_all, oof_s):.5f}')
    oof   += oof_s / len(LGB_SEEDS)
    tpred += tp_s  / len(LGB_SEEDS)

print(f'\nLightGBM multi-seed OOF AUC: {roc_auc_score(y_all, oof):.5f}')
np.save('oof_lgb.npy', oof); np.save('pred_lgb.npy', tpred)

In [ ]:
np.save('oof_lgb.npy', oof); np.save('pred_lgb.npy', tpred)
print(f'LightGBM OOF AUC: {roc_auc_score(y_all, oof):.5f}')

In [ ]:
from catboost import CatBoostClassifier

# CatBoost handles categoricals natively — pass cat_features indices
cat_cols = [c for c in CATS if c in X_all.columns]
cat_idx  = [X_all.columns.tolist().index(c) for c in cat_cols]

cb_params = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, random_seed=SEED,
    eval_metric='AUC', od_type='Iter', od_wait=100,
    verbose=False, thread_count=-1,
)

oof_cat  = np.zeros(len(X_all), dtype=np.float32)
pred_cat = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all.iloc[tri].copy();  Xval = X_all.iloc[vali].copy()
    ytr  = y_all.iloc[tri].copy();  yval = y_all.iloc[vali].copy()
    Xte  = Xtest_raw.copy()

    # CatBoost needs string cats — already string in X_all
    model_cb = CatBoostClassifier(**cb_params)
    model_cb.fit(Xtr, ytr, cat_features=cat_idx,
                 eval_set=(Xval, yval), use_best_model=True)

    oof_cat[vali] = model_cb.predict_proba(Xval)[:, 1]
    pred_cat += model_cb.predict_proba(Xte)[:, 1] / N_SPLITS

    print(f'CatBoost Fold {fold} AUC: {roc_auc_score(yval, oof_cat[vali]):.5f}  '
          f'best_iter={model_cb.best_iteration_}  time={time.time()-fold_start:.0f}s')

print(f'\nCatBoost OOF AUC: {roc_auc_score(y_all, oof_cat):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_cat.npy', oof_cat); np.save('pred_cat.npy', pred_cat)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Build vocabulary for each categorical ─────────────────────
# We need to know how many unique values each cat has to size the embedding layers.
# Use train+test combined so test categories don't get 'unknown' codes.
cat_vocabs = {}
for c in cat_cols:
    all_vals = pd.concat([X_all[c], Xtest_raw[c]], ignore_index=True).astype(str)
    vocab = {v: i+1 for i, v in enumerate(all_vals.unique())}  # 0 reserved for padding
    cat_vocabs[c] = vocab

def encode_cats(df, vocabs):
    out = {}
    for c, vocab in vocabs.items():
        out[c] = df[c].astype(str).map(vocab).fillna(0).astype('int64')
    return out

# Numeric columns = everything that isn't a cat
num_cols = [c for c in X_all.columns if c not in cat_cols]
print(f'{len(cat_cols)} cat cols, {len(num_cols)} numeric cols')

# ── PyTorch MLP with Embeddings ────────────────────────────────
class ChurnMLP(nn.Module):
    def __init__(self, cat_vocabs, num_dim, emb_dropout=0.2, hidden=[512, 256, 128], dropout=0.3):
        super().__init__()
        # Embedding layer per categorical
        # Rule of thumb: embedding dim = min(50, (n_categories+1)//2)
        self.embeddings = nn.ModuleDict({
            c: nn.Embedding(len(vocab)+1, min(50, (len(vocab)+1)//2))
            for c, vocab in cat_vocabs.items()
        })
        emb_total = sum(min(50, (len(v)+1)//2) for v in cat_vocabs.values())
        in_dim = emb_total + num_dim

        layers = [nn.BatchNorm1d(in_dim), nn.Dropout(emb_dropout)]
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        # Embed each categorical and concatenate
        embs = [self.embeddings[c](x_cat[:, i]) for i, c in enumerate(cat_cols)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.net(x).squeeze(1)

def make_tensors(Xdf, yvals=None):
    # cat integer codes → shape (N, n_cats)
    cat_enc = encode_cats(Xdf, cat_vocabs)
    x_cat = torch.tensor(
        np.stack([cat_enc[c].values for c in cat_cols], axis=1), dtype=torch.long
    ).to(device)
    x_num = torch.tensor(Xdf[num_cols].values.astype('float32')).to(device)
    if yvals is not None:
        y = torch.tensor(yvals.values.astype('float32')).to(device)
        return x_cat, x_num, y
    return x_cat, x_num

NN_EPOCHS   = 50
BATCH_SIZE  = 2048
LR          = 1e-3
PATIENCE    = 8

oof_nn  = np.zeros(len(X_all),     dtype=np.float32)
pred_nn = np.zeros(len(Xtest_raw), dtype=np.float32)
cv_start = time.time()

for fold, (tri, vali) in enumerate(skf.split(X_all, y_all), 1):
    fold_start = time.time()
    Xtr  = X_all.iloc[tri];   ytr  = y_all.iloc[tri]
    Xval = X_all.iloc[vali];  yval = y_all.iloc[vali]

    # Scale numerics — fit on train only, apply to val+test
    scaler = StandardScaler()
    Xtr_num  = scaler.fit_transform(Xtr[num_cols].values.astype('float32'))
    Xval_num = scaler.transform(Xval[num_cols].values.astype('float32'))
    Xte_num  = scaler.transform(Xtest_raw[num_cols].values.astype('float32'))

    # Build scaled dataframes for tensor creation
    Xtr_s  = Xtr.copy();  Xtr_s[num_cols]  = Xtr_num
    Xval_s = Xval.copy(); Xval_s[num_cols] = Xval_num
    Xte_s  = Xtest_raw.copy(); Xte_s[num_cols] = Xte_num

    xc_tr, xn_tr, y_tr = make_tensors(Xtr_s,  ytr)
    xc_val, xn_val, y_val_t = make_tensors(Xval_s, yval)
    xc_te, xn_te = make_tensors(Xte_s)

    train_ds = TensorDataset(xc_tr, xn_tr, y_tr)
    loader   = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    model_nn = ChurnMLP(cat_vocabs, len(num_cols)).to(device)
    optimizer = torch.optim.Adam(model_nn.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NN_EPOCHS)
    criterion = nn.BCEWithLogitsLoss()

    best_auc_nn = 0; patience_ct = 0; best_state = None

    for epoch in range(NN_EPOCHS):
        model_nn.train()
        for xc_b, xn_b, y_b in loader:
            optimizer.zero_grad()
            loss = criterion(model_nn(xc_b, xn_b), y_b)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model_nn.eval()
        with torch.no_grad():
            val_logits = model_nn(xc_val, xn_val).cpu().numpy()
            val_probs  = torch.sigmoid(torch.tensor(val_logits)).numpy()
        val_auc = roc_auc_score(yval, val_probs)

        if val_auc > best_auc_nn:
            best_auc_nn = val_auc; patience_ct = 0
            best_state = {k: v.clone() for k, v in model_nn.state_dict().items()}
        else:
            patience_ct += 1
            if patience_ct >= PATIENCE:
                break

    # Load best weights
    model_nn.load_state_dict(best_state)
    model_nn.eval()
    with torch.no_grad():
        oof_nn[vali] = torch.sigmoid(model_nn(xc_val, xn_val)).cpu().numpy()
        pred_nn += torch.sigmoid(model_nn(xc_te, xn_te)).cpu().numpy() / N_SPLITS

    print(f'NN Fold {fold} AUC: {roc_auc_score(yval, oof_nn[vali]):.5f}  '
          f'best_epoch={epoch-patience_ct+1}  time={time.time()-fold_start:.0f}s')

print(f'\nNN OOF AUC: {roc_auc_score(y_all, oof_nn):.5f}  total={time.time()-cv_start:.0f}s')
np.save('oof_nn.npy', oof_nn); np.save('pred_nn.npy', pred_nn)

In [ ]:
# ── Hill Climbing Ensemble ────────────────────────────────────
models = {
    'lgb': (oof,       tpred),
    'cat': (oof_cat,   pred_cat),
    'nn':  (oof_nn,    pred_nn),
}

print('Single model OOF AUCs:')
for name, (o, _) in models.items():
    print(f'  {name}: {roc_auc_score(y_all, o):.5f}')

best_name = max(models, key=lambda n: roc_auc_score(y_all, models[n][0]))
best_auc  = roc_auc_score(y_all, models[best_name][0])
print(f'\nStarting from: {best_name} (AUC={best_auc:.5f})')

weights = {best_name: 1.0}
blend_oof  = models[best_name][0].copy().astype(float)
blend_pred = models[best_name][1].copy().astype(float)

TOL = 1e-5
for step in range(20):
    best_step_auc = best_auc
    best_w = None; best_add = None
    for name, (o, _) in models.items():
        for w in np.arange(0.05, 1.0, 0.05):
            trial = (blend_oof + w * o) / (1 + w)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = w; best_add = name
        # try negative weights too
        for w in np.arange(0.05, 0.5, 0.05):
            trial = (blend_oof - w * o) / (1 - w)
            trial = np.clip(trial, 0, 1)
            auc = roc_auc_score(y_all, trial)
            if auc > best_step_auc:
                best_step_auc = auc; best_w = -w; best_add = name

    if best_add is None or (best_step_auc - best_auc) < TOL:
        print(f'Stopped: improvement {best_step_auc - best_auc:.8f} < TOL={TOL}')
        break

    w = best_w
    blend_oof  = (blend_oof  + w * models[best_add][0]) / (1 + w)
    blend_pred = (blend_pred + w * models[best_add][1]) / (1 + w)
    blend_oof  = np.clip(blend_oof, 0, 1)
    blend_pred = np.clip(blend_pred, 0, 1)
    weights[best_add] = weights.get(best_add, 0) + w
    best_auc = best_step_auc
    print(f'  Step {step}: AUC {best_auc:.6f} | add {best_add} w={w:.3f}')

print(f'\nFinal ensemble OOF AUC: {roc_auc_score(y_all, blend_oof):.5f}')
print('Weights:', weights)

sub['Churn'] = blend_pred
sub.to_csv('submission_ensemble.csv', index=False)
print('Saved submission_ensemble.csv')

In [ ]:
# ── Pseudo-labeling (Technique 6) ────────────────────────────
# Use high-confidence ensemble predictions as soft labels on test set
PSEUDO_THRESH = 0.15  # confident no-churn: <0.15, confident churn: >0.85
mask = (blend_pred < PSEUDO_THRESH) | (blend_pred > (1 - PSEUDO_THRESH))
print(f'High-confidence test samples: {mask.sum():,} / {len(blend_pred):,} ({mask.mean()*100:.1f}%)')

# Build expanded training set
X_pseudo = pd.concat([X_all, Xtest_raw[mask].reset_index(drop=True)], ignore_index=True)
y_pseudo  = np.concatenate([y_all.values, blend_pred[mask]])  # soft labels

print(f'Expanded training set: {len(X_pseudo):,} rows')

# Retrain LGB on full train + pseudo-labeled test (fixed iters — no val set)
# Use best_iteration from previous LGB run × 1.1 as fixed n_estimators
best_iters = model.best_iteration_ if hasattr(model, 'best_iteration_') else 400
n_est = int(best_iters * 1.1)
print(f'Retraining LGB on full data + pseudo labels, n_estimators={n_est}...')

pseudo_params = {**lgb_params, 'n_estimators': n_est, 'random_state': SEED}

oof_pseudo  = np.zeros(len(X_all),     dtype=np.float64)
pred_pseudo = np.zeros(len(Xtest_raw), dtype=np.float64)

for seed in LGB_SEEDS:
    Xp = X_pseudo.copy(); Xte = Xtest_raw.copy()
    for c in CATS:
        if c in Xp.columns:
            codes, _ = pd.factorize(pd.concat([Xp[c], Xte[c]], ignore_index=True).astype(str))
            Xp[c]  = codes[:len(Xp)].astype('int32')
            Xte[c] = codes[len(Xp):].astype('int32')
    for c in Xp.select_dtypes('object').columns:
        codes, _ = pd.factorize(pd.concat([Xp[c], Xte[c]], ignore_index=True).astype(str))
        Xp[c]  = codes[:len(Xp)].astype('int32')
        Xte[c] = codes[len(Xp):].astype('int32')
    m = lgb.LGBMClassifier(**{**pseudo_params, 'random_state': seed})
    m.fit(Xp, y_pseudo)
    pred_pseudo += m.predict_proba(Xte)[:, 1] / len(LGB_SEEDS)

# Blend pseudo predictions with ensemble
PSEUDO_WEIGHT = 0.3
final_pred = (1 - PSEUDO_WEIGHT) * blend_pred + PSEUDO_WEIGHT * pred_pseudo
print(f'Pseudo-label blend weight: {PSEUDO_WEIGHT}')

sub['Churn'] = final_pred
sub.to_csv('submission_final.csv', index=False)
print('Saved submission_final.csv')